# Module 6: All Beauty cleaning
Business question: Which parent listings have high negative (1-2 star) review shares, and how much captured listing content is absent? Review counts do not represent sales. Metadata is a snapshot; store is not necessarily brand.

One row is one retained source reviewer-product record joined to a parent listing. Repeated IDs and user-product-time combinations are not asserted unique.

In [1]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
ROOT = Path.cwd()
assert (ROOT / "config/cleaning_rules.json").exists()
rules = json.loads((ROOT / "config/cleaning_rules.json").read_text())
print("Interpreter:", sys.executable)
print("Project root:", ROOT)
for name, rule in rules["rules"].items():
    print(name + ": " + rule)

Interpreter: /Users/hcx/dso576/.venv/bin/python
Project root: /Users/hcx/dso576/DSO-576-
duplicates: SHA-256 of sorted-key canonical JSON of ALL raw fields including images; keep first source occurrence. No deletion for repeated IDs.
text: Preserve originals; derived text strips outer whitespace, blank to null; preserve case, punctuation, HTML, interior whitespace and encoding. Never normalize IDs.
price: Preserve raw; strip outer whitespace, None/empty to null; numeric parse failures flagged; negative/nonfinite to null and flagged; zero and >1000 retained and flagged.
store: Null stays null; flag missing; no deletion; store is not assumed brand.
timestamp: Preserve epoch milliseconds; UTC derived time; flag missing, failed, and outside May 1996 through September 2023.
validation: Derived rating integer 1-5, helpful vote nonnegative integer, verified purchase actual boolean; otherwise null and flag.
content: Retain arrays and details raw; ordered newline text; parsed details serialized

## Execute the reusable raw-input pipeline
Both original hashes are checked before and after processing. Duplicate fingerprints include every raw field, including images. Derived text preserves HTML/interior whitespace; original fields remain available. Invalid derived values become null with flags. Metadata keys must be unique and the left join must be many-to-one. All supplied expectations must pass before full-table exports.

In [2]:
subprocess.run([sys.executable, str(ROOT / "scripts/clean_data.py")], cwd=ROOT, check=True)
audit = json.loads((ROOT / "outputs/reports/audit.json").read_text())
assert audit["status"] == "PASS"
assert audit["source_sha256_before"] == audit["source_sha256_after"]
display(pd.Series(audit["counts"], name="observed").to_frame())
display(pd.Series(audit["metrics"], name="observed").to_frame())
print("UTC range:", audit["timestamp_min"], audit["timestamp_max"])

{
  "status": "PASS",
  "counts": {
    "raw_reviews": 701528,
    "duplicate_copies_removed": 7275,
    "duplicate_groups": 6138,
    "retained_reviews": 694253,
    "metadata_records": 112590,
    "merged_rows": 694253,
    "unmatched_retained_reviews": 0,
    "unmatched_parent_keys": 0,
    "extra_merge_rows": 0,
    "duplicate_metadata_keys": 0,
    "metadata_without_retained_review": 25,
    "raw_rating_sum": 2778223.0,
    "duplicate_rating_sum": 28361.0,
    "retained_rating_sum": 2749862.0,
    "raw_helpful_total": 647923,
    "duplicate_helpful_total": 6313,
    "retained_helpful_total": 641610.0,
    "review_title_changes": 46,
    "review_body_changes": 746,
    "blank_review_bodies": 705,
    "products_missing_price": 94886,
    "reviews_missing_price": 510545,
    "price_parse_failures": 0,
    "reviews_price_over_1000": 13,
    "reviews_missing_store": 49380,
    "timestamp_failed": 0,
    "timestamp_outside_window": 0,
    "rating_invalid": 0,
    "helpful_vote_invalid":

,observed
blank_review_bodies,705.0
categories_missing,112590.0
description_missing,93428.0
details_missing,4512.0
details_parse_failures,0.0
duplicate_copies_removed,7275.0
duplicate_groups,6138.0
duplicate_helpful_total,6313.0
duplicate_metadata_keys,0.0
duplicate_rating_sum,28361.0


,observed
helpful_vote_denominator,694253.000000
mean_available_product_price,27.255730
mean_review_rating,3.960893
negative_review_rate,0.206681
rating_denominator,694253.000000
verified_denominator,694253.000000
verified_purchase_rate,0.905226


UTC range: 2000-11-01 04:24:18+00:00 2023-09-09 00:39:36.666000+00:00


## Five predeclared checks
Expectations were written before the first cleaning run. Row 6567 is excluded; row 6566 supplies retained counterpart evidence in the sample.

In [3]:
checks = json.loads((ROOT / "outputs/reports/record_checks.json").read_text())
assert len(checks) == 5 and all(c["pass"] for c in checks)
display(pd.DataFrame(checks))
sample = pd.read_csv(ROOT / "outputs/data/cleaned_sample.csv")
assert len(sample) == 50 and 6567 not in sample.source_review_row.values
display(sample[["source_review_row", "review_title_clean", "price_numeric", "store_missing", "selection_reason"]].head(12))

,actual,expected,field,id,original,pass,reason,source_review_row
0,"{'original': 'None', 'price_numeric': None}",None,price_numeric,A,None,True,Literal None means unknown price.,1
1,"{'original': '9.99', 'price_numeric': 9.99}",9.99,price_numeric,B,9.99,True,Valid numeric string.,31
2,"{'retained': True, 'store_missing': True}","{'retained': True, 'store_missing': True}",NaN,C,NaN,True,Missing store does not remove reviews.,28
3,"{'cleaned_length': 819, 'only_trailing_nbsp_re...","{'cleaned_length': 819, 'only_trailing_nbsp_re...",NaN,D,NaN,True,Only outer whitespace changes; interior text a...,213
4,"{'counterpart_retained': True, 'logged_first_r...","{'counterpart_retained': True, 'logged_first_r...",NaN,E,NaN,True,Exact full-raw duplicate of first occurrence 6...,6567


,source_review_row,review_title_clean,price_numeric,store_missing,selection_reason
0,1,Such a lovely scent but not overpowering.,NaN,False,predeclared missing price
1,2,Works great but smells a little weird.,NaN,False,first flag: description_missing
2,28,not what it claims to do,NaN,True,predeclared missing store
3,31,Ineffective,9.99,False,predeclared numeric price
4,123,I recommend this product to other customers,3.99,False,first flag: details_missing
5,213,PU case really smells; brushes are okay; two b...,NaN,False,predeclared trailing NBSP
6,6566,Five Stars,NaN,False,retained duplicate counterpart of excluded row...
7,14634,Great for small vanilla batches,13.99,False,deterministic random seed 576
8,16054,GREAT SHOWER CAP,NaN,False,deterministic random seed 576
9,38483,Continue to love Proactive cleanser,18.99,False,deterministic random seed 576


## Reproducibility and limitations
The verification script regenerates cleaning artifacts in isolated temporary output storage. It compares logical Parquet tables and exact hashes for JSON/CSV. Price mean uses unique metadata products; review metrics use valid denominators. Missing captured listing content does not establish absence from live listings. A personal independent check and final Git submission metadata remain manual tasks.

In [4]:
repro = json.loads((ROOT / "outputs/reports/reproducibility.json").read_text())
assert repro["status"] == "PASS"
display(pd.DataFrame(repro["compared"]))
print(json.loads((ROOT / "config/submission.json").read_text()))

,artifact,method,pass
0,data/beauty_cleaned.parquet,"exact logical table equality, including order/...",True
1,data/cleaned_sample.csv,exact SHA-256,True
2,data/product_metrics.parquet,"exact logical table equality, including order/...",True
3,data/products_cleaned.parquet,"exact logical table equality, including order/...",True
4,reports/audit.json,exact SHA-256,True
5,reports/cleaned_types.csv,exact SHA-256,True
6,reports/missing_counts.csv,exact SHA-256,True
7,reports/record_checks.json,exact SHA-256,True
8,reports/removed_duplicates.csv,exact SHA-256,True
9,reports/source_profile.json,exact SHA-256,True


{'branch_name': 'alex', 'branch_url': None, 'analysis_commit_id': None, 'submitted_commit_id': None, 'remote_url': 'https://github.com/chihhsin1209/DSO-576-', 'status': 'INCOMPLETE', 'notes': 'Branch URL awaits remote verification after authorized push. Analysis commit identifies code/results; submitted commit identifies final submission. Neither is the starter HEAD. Do not claim submitted status before submission.'}
